# Engineer Holiday Features

Create holiday-related features for the ML training data to capture the effects of increased demand periods. This is motivated by the observations made that weather features have very weak correlations with delay rates.

**Features to engineer:**
- `n_public_holidays_{state}`: Count of public holidays per state/territory (ACT, NSW, NT, QLD, SA, TAS, VIC, WA)
- `n_public_holidays_national`: Count of national public holidays
- `n_public_holidays_total`: Total unique public holidays across all states
- `has_major_holiday`: Binary flag for major holiday periods (Christmas, Easter, ANZAC)
- `school_holiday_days`: Days overlapping school holidays
- `pct_school_holiday`: Fraction of month in school holidays

**Data sources:**
- Public holidays: Python `holidays` library
- School holidays: Manual approximation

In [ ]:
import pandas as pd
import numpy as np
from datetime import date, timedelta
from calendar import monthrange

# Install if needed: pip install holidays
import holidays

In [ ]:
# Load existing ML training data to get date range
df = pd.read_csv('../data/processed/ml_training_data_syd_mel.csv')

# Fix column types - 'month' column contains dates, extract actual month number
df['month_num'] = pd.to_datetime(df['month']).dt.month
df['year'] = df['year'].astype(int)

print(f"Data shape: {df.shape}")
print(f"Date range: {df['year_month'].min()} to {df['year_month'].max()}")

In [ ]:
# Get unique year-months
year_months = df[['year', 'month_num']].drop_duplicates().sort_values(['year', 'month_num'])
print(f"Unique months to process: {len(year_months)}")

## 1. Public Holidays (using `holidays` library)

It is important to note that each state and territory has their own holidays in addition to the national holidays.

In [ ]:
# Create holiday calendars for all Australian states and territories
years = list(range(2010, 2026))

# All state/territory codes
STATES = ['ACT', 'NSW', 'NT', 'QLD', 'SA', 'TAS', 'VIC', 'WA']

# Create holiday dict for each state
state_holidays = {}
for state in STATES:
    state_holidays[state] = holidays.Australia(years=years, prov=state)
    print(f"{state} holidays loaded: {len(state_holidays[state])}")

# Also create national holidays (no state specified)
national_holidays = holidays.Australia(years=years)
print(f"\nNational holidays loaded: {len(national_holidays)}")

Have a quick look of how the holiday data is presented.

In [ ]:
# Preview some holidays by state (2024)
print("Sample holidays by state (2024):\n")
for state in ['NSW', 'VIC', 'QLD']:  # Show a few states as examples
    print(f"--- {state} ---")
    for d, name in sorted(state_holidays[state].items()):
        if d.year == 2024:
            print(f"  {d}: {name}")
    print()

Create separate functions for:
* counting number of holidays in a specific month year (integer)
* classifying if a month contains a major holiday (binary)

In [ ]:
def count_holidays_in_month(holiday_dict, year, month):
    """Count number of public holidays in a given month."""
    count = 0
    _, days_in_month = monthrange(year, month)
    # Loop through each day of the month to check if holiday
    for day in range(1, days_in_month + 1):
        if date(year, month, day) in holiday_dict:
            count += 1
    return count

def has_major_holiday_in_month(holiday_dict, year, month):
    """Check if month contains major holidays (Christmas, Easter, ANZAC)."""
    major_keywords = ['christmas', 'easter', 'anzac', 'good friday', 'boxing']
    _, days_in_month = monthrange(year, month)
    # Loop through each day of the month to check if holiday name is in major_keywords
    for day in range(1, days_in_month + 1):
        d = date(year, month, day)
        if d in holiday_dict:
            name = holiday_dict[d].lower()
            if any(kw in name for kw in major_keywords):
                return 1
    return 0

In [ ]:
# Compute public holiday features for each month (all states)
public_holiday_features = []

for _, row in year_months.iterrows():
    year, month = int(row['year']), int(row['month_num'])
    
    feature_row = {
        'year': year,
        'month_num': month,
    }
    
    # Count holidays for each state
    all_holiday_dates = set()
    for state in STATES:
        n_state = count_holidays_in_month(state_holidays[state], year, month)
        feature_row[f'n_public_holidays_{state.lower()}'] = n_state
        
        # Collect all holiday dates for total count
        _, days_in_month = monthrange(year, month)
        for day in range(1, days_in_month + 1):
            d = date(year, month, day)
            if d in state_holidays[state]:
                all_holiday_dates.add(d)
    
    # National holidays count
    feature_row['n_public_holidays_national'] = count_holidays_in_month(national_holidays, year, month)
    
    # Total unique holidays across all states
    feature_row['n_public_holidays_total'] = len(all_holiday_dates)
    
    # Check for major holidays (using any state's calendar)
    has_major = 0
    for state in STATES:
        if has_major_holiday_in_month(state_holidays[state], year, month):
            has_major = 1
            break
    feature_row['has_major_holiday'] = has_major
    
    public_holiday_features.append(feature_row)

df_public_holidays = pd.DataFrame(public_holiday_features)
print(f"Shape: {df_public_holidays.shape}")
print(f"Columns: {list(df_public_holidays.columns)}")
df_public_holidays.head(15)

In [ ]:
# Verify major holidays are flagged correctly
print("Months with major holidays:")
print(df_public_holidays[df_public_holidays['has_major_holiday'] == 1].groupby('month_num').size())

# Show variation across states
print("\nAverage public holidays per month by state:")
state_cols = [f'n_public_holidays_{s.lower()}' for s in STATES]
print(df_public_holidays[state_cols].mean().round(2))

## 2. School Holidays

Australian school holidays vary by state and year. We'll use approximate dates that capture the main patterns:

- **Term 1 break (Easter):** ~2 weeks in April
- **Term 2 break (Winter):** ~2 weeks in late June/July
- **Term 3 break (Spring):** ~2 weeks in late September/early October
- **Term 4 break (Summer):** ~6 weeks mid-December to late January

Note: Exact dates shift each year. For simplicity, we use approximate windows that capture peak travel periods.

In [ ]:
# Define approximate school holiday windows (month, start_day, end_day)
# These are simplified - actual dates vary by ~1 week each year

def get_school_holiday_periods(year):
    """
    Return list of (start_date, end_date) tuples for school holidays.
    Uses approximate dates that capture peak travel periods.
    """
    periods = [
        # Summer holidays (Dec-Jan, spans two years)
        (date(year-1, 12, 18), date(year, 1, 28)),
        # Easter/Autumn break (April)
        (date(year, 4, 8), date(year, 4, 23)),
        # Winter break (June-July)
        (date(year, 6, 27), date(year, 7, 14)),
        # Spring break (Sept-Oct)
        (date(year, 9, 23), date(year, 10, 7)),
        # Summer holidays start (December of current year)
        (date(year, 12, 18), date(year, 12, 31)),
    ]
    return periods

# Test
print("School holiday periods for 2024:")
for start, end in get_school_holiday_periods(2024):
    print(f"  {start} to {end}")

In [ ]:
def count_school_holiday_days_in_month(year, month):
    """
    Count days in a month that fall within school holiday periods.
    """
    periods = get_school_holiday_periods(year)
    # Also check previous year's summer holidays for January
    if month == 1:
        periods.extend(get_school_holiday_periods(year))
    
    _, days_in_month = monthrange(year, month)
    holiday_days = 0
    
    for day in range(1, days_in_month + 1):
        current_date = date(year, month, day)
        for start, end in periods:
            if start <= current_date <= end:
                holiday_days += 1
                break  # Don't double count
    
    return holiday_days

In [ ]:
# Compute school holiday features
school_holiday_features = []

for _, row in year_months.iterrows():
    year, month = int(row['year']), int(row['month_num'])
    _, days_in_month = monthrange(year, month)
    
    school_days = count_school_holiday_days_in_month(year, month)
    
    school_holiday_features.append({
        'year': year,
        'month_num': month,
        'school_holiday_days': school_days,
        'pct_school_holiday': round(school_days / days_in_month, 4)
    })

df_school_holidays = pd.DataFrame(school_holiday_features)
print(df_school_holidays.head(15))

In [ ]:
# Verify school holiday distribution by month
print("Average school holiday days by month:")
print(df_school_holidays.groupby('month_num')['school_holiday_days'].mean().round(1))

## 3. Combine All Holiday Features

In [ ]:
# Merge public and school holiday features
df_holidays = df_public_holidays.merge(df_school_holidays, on=['year', 'month_num'])
print(f"Holiday features shape: {df_holidays.shape}")
df_holidays.head(10)

In [ ]:
# Summary statistics
df_holidays.describe()

## 4. Merge with ML Training Data

In [ ]:
# Merge holiday features into training data
df_merged = df.merge(df_holidays, on=['year', 'month_num'], how='left')

print(f"Original shape: {df.shape}")
print(f"Merged shape: {df_merged.shape}")
print(f"New columns added: {df_merged.shape[1] - df.shape[1]}")

In [ ]:
# Check for any missing values in new columns
state_holiday_cols = [f'n_public_holidays_{s.lower()}' for s in STATES]
holiday_cols = state_holiday_cols + ['n_public_holidays_national', 'n_public_holidays_total',
                                      'has_major_holiday', 'school_holiday_days', 'pct_school_holiday']
print("Missing values in holiday features:")
print(df_merged[holiday_cols].isnull().sum())

In [ ]:
# Preview merged data
preview_cols = ['year_month', 'airline', 'delay_rate', 'n_public_holidays_nsw', 'n_public_holidays_vic', 
                'n_public_holidays_qld', 'n_public_holidays_total', 'has_major_holiday', 
                'school_holiday_days', 'pct_school_holiday']
df_merged[preview_cols].head(10)

## 5. Quick Validation: Holiday Features vs Delay Rate

In [ ]:
# Correlation with target
print("Correlation with delay_rate:\n")
print("By state:")
for state in STATES:
    col = f'n_public_holidays_{state.lower()}'
    r = df_merged[col].corr(df_merged['delay_rate'])
    print(f"  {col}: {r:.4f}")

print("\nAggregate features:")
for col in ['n_public_holidays_national', 'n_public_holidays_total', 'has_major_holiday', 
            'school_holiday_days', 'pct_school_holiday']:
    r = df_merged[col].corr(df_merged['delay_rate'])
    print(f"  {col}: {r:.4f}")

Expected results for SYD - MEL route:
```
Correlation with delay_rate:

By state:
  n_public_holidays_act: -0.0131
  n_public_holidays_nsw: -0.0009
  n_public_holidays_nt: -0.0609
  n_public_holidays_qld: -0.0296
  n_public_holidays_sa: -0.0105
  n_public_holidays_tas: -0.0169
  n_public_holidays_vic: 0.0152
  n_public_holidays_wa: -0.0242

Aggregate features:
  n_public_holidays_national: -0.0080
  n_public_holidays_total: -0.0323
  has_major_holiday: 0.0397
  school_holiday_days: -0.0090
  pct_school_holiday: -0.0096
```

In [ ]:
# Delay rate by major holiday flag
print("\nDelay rate by has_major_holiday:")
print(df_merged.groupby('has_major_holiday')['delay_rate'].agg(['mean', 'std', 'count']))

Expected results for SYD - MEL route:

```
Delay rate by has_major_holiday:
                       mean       std  count
has_major_holiday                           
0                  0.239409  0.112153   1176
1                  0.250903  0.115159    270
```

In [ ]:
# Delay rate by school holiday percentage bins
df_merged['school_holiday_bin'] = pd.cut(df_merged['pct_school_holiday'], 
                                          bins=[0, 0.1, 0.3, 0.5, 1.0],
                                          labels=['0-10%', '10-30%', '30-50%', '50-100%'])
print("\nDelay rate by school holiday percentage:")
print(df_merged.groupby('school_holiday_bin', observed=False)['delay_rate'].agg(['mean', 'std', 'count']))

Expected results:

```
Delay rate by school holiday percentage:
                        mean       std  count
school_holiday_bin                           
0-10%                    NaN       NaN      0
10-30%              0.242520  0.111284    354
30-50%              0.274148  0.117456    230
50-100%             0.227329  0.113140    256
```

From observing these results:
* Holiday periods have little to effects on the flight delays on this route (at least not at the monthly aggregate level).
* Airlines may anticipate demand during holidays and schedule appropriately. Holidays are more predictable than weather conditions.
* However, it could be route specific. SYD - MEL route is predominantly business travel, thus holiday traffic may be a smaller proportion than expected.

## 6. Save Updated Data

In [ ]:
# Drop temporary column
df_merged = df_merged.drop(columns=['school_holiday_bin'])

# Save to new file
output_path = '../data/processed/ml_training_data_syd_mel_with_holidays.csv'
df_merged.to_csv(output_path, index=False)
print(f"Saved to: {output_path}")
print(f"Final shape: {df_merged.shape}")

In [ ]:
# Final column list
print("All columns in output:")
for i, col in enumerate(df_merged.columns):
    print(f"  {i+1}. {col}")

## Summary

**Holiday features added:**

*Public holidays (by state/territory):*
1. `n_public_holidays_act` - ACT public holiday count
2. `n_public_holidays_nsw` - NSW public holiday count
3. `n_public_holidays_nt` - NT public holiday count
4. `n_public_holidays_qld` - QLD public holiday count
5. `n_public_holidays_sa` - SA public holiday count
6. `n_public_holidays_tas` - TAS public holiday count
7. `n_public_holidays_vic` - VIC public holiday count
8. `n_public_holidays_wa` - WA public holiday count

*Aggregate public holidays:*
9. `n_public_holidays_national` - National public holiday count
10. `n_public_holidays_total` - Total unique holidays across all states

*Other features:*
11. `has_major_holiday` - Binary flag for Christmas/Easter/ANZAC periods
12. `school_holiday_days` - Days in school holiday periods
13. `pct_school_holiday` - Fraction of month in school holidays

**Output:** `data/processed/ml_training_data_syd_mel_with_holidays.csv`

**Note on school holidays:** The dates used are approximate averages. For more accuracy, historical school term dates could be sourced from state education departments.